# Clasificador de gestos: MLP vs. reglas

Secciones 5.2, 12 y 15 de la propuesta. El notebook:

1. Carga los CSV de `ml/data/` (grabados con `frontend/captura.html`).
2. Normaliza los landmarks con el mismo criterio que el navegador.
3. Evalúa **por persona**: cada persona se usa una vez como prueba y el modelo se entrena con las demás. Si se mezclan fotogramas de la misma persona en entrenamiento y prueba, la métrica sale inflada (sección 12.2).
4. Compara las **reglas del visor** con **dos variantes de MLP** sobre los mismos datos.
5. Entrena el modelo final y lo exporta a `ml/gesture_model.onnx` para usarlo en el navegador.

Las 5 clases son posturas de una mano. El escalado con dos manos se obtiene combinando las dos manos (las dos en pellizco), no es una clase del MLP.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, recall_score
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold
from sklearn.neural_network import MLPClassifier

RAIZ = Path.cwd().parent if Path.cwd().name == "ml" else Path.cwd()
ML = RAIZ / "ml"
sys.path.insert(0, str(ML))
from preparar import CLASES, cargar, normalizar  # noqa: E402

# Se pueden cambiar con variables de entorno (por ejemplo, para probar con datos sintéticos)
DATOS = Path(os.getenv("DATASET_DIR", ML / "data"))
RESULTADOS = Path(os.getenv("RESULTADOS_DIR", ML / "resultados"))
MODELO = Path(os.getenv("MODELO_SALIDA", ML / "gesture_model.onnx"))
SEMILLA = 42
RESULTADOS.mkdir(parents=True, exist_ok=True)
print("Datos:", DATOS)

## 1. Datos

In [ ]:
df = cargar(DATOS)
conteo = pd.crosstab(df["clase"], df["persona"], margins=True, margins_name="total")
display(conteo.reindex([c for c in CLASES if c in conteo.index] + ["total"]))

personas = sorted(df["persona"].unique())
if len(personas) < 4:
    print(f"ATENCIÓN: hay {len(personas)} persona(s). La sección 12.2 pide al menos 4 o 5 para que la evaluación sea confiable.")
faltan = set(CLASES) - set(df["clase"])
if faltan:
    print(f"ATENCIÓN: faltan clases en el dataset: {sorted(faltan)}")
print("Mano:", df["mano"].value_counts(dropna=False).to_dict())

## 2. Normalización

Igual que `frontend/js/normalize.js` (lo verifica `ml/test_preparar.py`): corrección de aspecto,
origen en la muñeca y escala muñeca → nudillo medio. Resultado: 63 valores por fotograma.

In [ ]:
X = normalizar(df)
y = df["clase"].to_numpy()
grupos = df["persona"].to_numpy()
print("X:", X.shape)

## 3. Evaluación por persona: reglas vs. MLP

- **Reglas**: las del visor (`frontend/js/gestures.js`), aplicadas fotograma a fotograma con `ml/evaluar_reglas.mjs`. No se entrenan.
- **MLP**: dos variantes (sección 12.4: "probar al menos dos variantes"). ReLU, Adam y entropía cruzada.
  scikit-learn no tiene *dropout*: el sobreajuste se controla con regularización L2 (`alpha`) y *early stopping*.

In [ ]:
salida_reglas = RESULTADOS / "reglas_predicciones.csv"
subprocess.run(["node", str(ML / "evaluar_reglas.mjs"), str(DATOS), str(salida_reglas)], cwd=RAIZ, check=True)
reglas = pd.read_csv(salida_reglas)
pred_reglas = df[["archivo", "fila"]].merge(reglas, on=["archivo", "fila"], how="left")["prediccion"].to_numpy()

BASE = dict(activation="relu", solver="adam", alpha=1e-3, max_iter=500,
            early_stopping=True, n_iter_no_change=20, random_state=SEMILLA)
VARIANTES = {
    "MLP 64-32": dict(hidden_layer_sizes=(64, 32)),
    "MLP 128-64": dict(hidden_layer_sizes=(128, 64)),
}

if len(personas) >= 2:
    particion = list(LeaveOneGroupOut().split(X, y, grupos))
else:
    print("ATENCIÓN: con una sola persona no se puede separar por persona. "
          "Se usa validación cruzada por fotograma: la métrica va a salir inflada.")
    particion = list(StratifiedKFold(5, shuffle=True, random_state=SEMILLA).split(X, y))


def evaluar(params):
    pred = np.empty(len(y), dtype=object)
    for entrenamiento, prueba in particion:
        modelo = MLPClassifier(**BASE, **params).fit(X[entrenamiento], y[entrenamiento])
        pred[prueba] = modelo.predict(X[prueba])
    return pred


predicciones = {"Reglas": pred_reglas}
for nombre, params in VARIANTES.items():
    predicciones[nombre] = evaluar(params)
    print(f"{nombre}: listo")

In [ ]:
filas = []
for nombre, pred in predicciones.items():
    fila = {"método": nombre, "total": accuracy_score(y, pred)}
    for p in personas:
        m = grupos == p
        fila[f"persona {p}"] = accuracy_score(y[m], pred[m])
    filas.append(fila)
comparacion = pd.DataFrame(filas).set_index("método")
comparacion.to_csv(RESULTADOS / "comparacion_por_persona.csv")
print("Exactitud (%) — cada persona evaluada con un modelo que no la vio:")
display((comparacion * 100).round(1))

por_clase = pd.DataFrame({nombre: recall_score(y, pred, labels=CLASES, average=None, zero_division=0)
                          for nombre, pred in predicciones.items()}, index=CLASES)
por_clase.to_csv(RESULTADOS / "comparacion_por_clase.csv")
print("Aciertos por gesto (%) — sección 15: precisión por gesto:")
display((por_clase * 100).round(1))

ax = (comparacion["total"] * 100).plot.bar(color=["#8fb3c4", "#2bd4ff", "#1a8fb0"], rot=0, figsize=(6, 3.5))
ax.set_ylabel("Exactitud (%)")
ax.set_ylim(0, 110)
ax.set_xlabel("")
ax.set_title("Reglas vs. MLP (evaluación por persona)", pad=12)
for barra in ax.patches:
    ax.annotate(f"{barra.get_height():.0f}%", (barra.get_x() + barra.get_width() / 2, barra.get_height()),
                ha="center", va="bottom")
plt.tight_layout()
plt.savefig(RESULTADOS / "reglas_vs_mlp.png", dpi=150)
plt.show()

In [ ]:
mejor = comparacion.drop(index="Reglas")["total"].idxmax()
fig, ejes = plt.subplots(1, 2, figsize=(11, 4.5))
for eje, nombre in zip(ejes, ["Reglas", mejor]):
    ConfusionMatrixDisplay.from_predictions(y, predicciones[nombre], labels=CLASES, ax=eje,
                                            colorbar=False, normalize="true", values_format=".0%")
    eje.set_title(nombre)
    eje.set_xlabel("Gesto predicho")
    eje.set_ylabel("Gesto real")
    eje.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(RESULTADOS / "matrices_confusion.png", dpi=150)
plt.show()

## 4. Modelo final

Se entrena la mejor variante con todos los datos. Las curvas sirven para detectar sobreajuste
(sección 12.4): si la pérdida de entrenamiento baja pero la exactitud de validación no sube, el modelo memoriza.

In [ ]:
final = MLPClassifier(**BASE, **VARIANTES[mejor]).fit(X, y)
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.5))
a.plot(final.loss_curve_)
a.set_title("Pérdida de entrenamiento")
a.set_xlabel("época")
b.plot(final.validation_scores_)
b.set_title("Exactitud de validación (10 % de los datos)")
b.set_xlabel("época")
plt.tight_layout()
plt.savefig(RESULTADOS / "curvas_entrenamiento.png", dpi=150)
plt.show()
print(f"Variante: {mejor} · épocas: {final.n_iter_}")

## 5. Exportación a ONNX (para usar el modelo en el navegador)

In [ ]:
import onnxruntime as ort
from skl2onnx import to_onnx

modelo_onnx = to_onnx(final, X[:1], options={id(final): {"zipmap": False}})
MODELO.write_bytes(modelo_onnx.SerializeToString())

# Verificación: el modelo exportado tiene que dar lo mismo que el de scikit-learn
sesion = ort.InferenceSession(str(MODELO))
etiquetas, probabilidades = sesion.run(None, {sesion.get_inputs()[0].name: X[:500]})
assert (etiquetas == final.predict(X[:500])).all()
np.testing.assert_allclose(probabilidades, final.predict_proba(X[:500]), atol=1e-4)

metadatos = {
    "clases": [str(c) for c in final.classes_],
    "entrada": "63 valores: 21 landmarks (x, y, z) normalizados como frontend/js/normalize.js",
    "variante": mejor,
    "exactitud_por_persona": round(float(comparacion.loc[mejor, "total"]), 4),
    "exactitud_reglas": round(float(comparacion.loc["Reglas", "total"]), 4),
    "personas": [str(p) for p in personas],
    "muestras": int(len(X)),
}
MODELO.with_suffix(".json").write_text(json.dumps(metadatos, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Exportado: {MODELO} ({MODELO.stat().st_size / 1024:.0f} KB)")
print(json.dumps(metadatos, ensure_ascii=False, indent=2))

## 6. Conclusiones (completar con los datos reales)

Guía para la defensa (sección 5.2, "Por qué la comparación importa"):

- ¿Cuántos puntos mejora el MLP a las reglas? ¿En qué gestos?
- ¿Qué gestos confunde cada método? (matrices de confusión)
- ¿Funciona igual con todas las personas o hay alguna con peores resultados? ¿Por qué podría ser? (tamaño de mano, luz, mano izquierda)
- ¿Vale la pena el costo de juntar datos frente a la explicabilidad de las reglas?